# llama3-mhc: mHC from zero

A short notebook to *see* what mHC (Manifold-Constrained Hyper-Connections, [arXiv:2512.24880](https://arxiv.org/abs/2512.24880)) does inside a Llama-3-style backbone:

1. the residual stream is expanded into **n parallel streams**;
2. per-token **annotations** (read / post / residual) come from one projection of the flattened stream (Eq.7);
3. the residual mixing matrix is projected onto **doubly stochastic matrices** by Sinkhorn-Knopp (Eq.8-9, 20 iters) — this is what keeps signals bounded.

Run on CPU is fine (small model).

In [ ]:
import torch
from llama3_mhc import LlamaHC, LlamaHCConfig
from llama3_mhc.mixers import SinkhornMHCResidual, doubly_stochastic_error

cfg = LlamaHCConfig(n_layer=2, n_head=4, n_kv_heads=2, n_embd=128,
                    block_size=32, vocab_size=65, mixer="sinkhorn")
m = LlamaHC(cfg).eval()
print(f"params: {sum(p.numel() for p in m.parameters())/1e6:.2f}M, n_streams={cfg.n_streams}")

## 1. Stream expansion

The embedding is **replicated** across `n=4` streams (official `expand_to_mhc` convention).

In [ ]:
x = torch.randint(0, 65, (1, 8))
tok = m.transformer.wte(x)
streams = tok.unsqueeze(2).expand(-1, -1, cfg.n_streams, -1).contiguous()
print("streams shape (B, T, n, d):", tuple(streams.shape))
print("all streams identical at init:", torch.allclose(streams[:, :, 0], streams[:, :, -1]))

## 2. Per-token annotations (Eq.7)

For each stage, one linear projection of the **flattened** stream RMSNorm gives read `h_pre`, post `h_post` (2-sigmoid), and the residual raw matrix; `h_post` init = 0 means the block starts as near-identity.

In [ ]:
blk = m.transformer.h[0]
h_pre, h_post, raw = blk._dyn_raw(streams, "attn")
print("h_pre  range:", float(h_pre.min()), "-", float(h_pre.max()))   # ~ (0,1)
print("h_post range:", float(h_post.min()), "-", float(h_post.max())) # ~ (0,2), init ~1
print("raw shape:", tuple(raw.shape), "(B, T, n, n)")

## 3. Sinkhorn -> doubly stochastic (Eq.8-9)

20 alternating row/col normalizations project the raw matrix onto the Birkhoff polytope: rows **and** cols sum to 1, entries non-negative.

In [ ]:
import torch.nn.functional as F
W = F.softmax(torch.randn(1, 1, 4, 4), dim=-1)  # a random non-DS matrix
print("before Sinkhorn row/col err:", doubly_stochastic_error(W))

def sinkhorn(H, iters=20):
    H = torch.exp(H)
    for _ in range(iters):
        H = H / H.sum(-1, keepdim=True).clamp_min(1e-12)
        H = H / H.sum(-2, keepdim=True).clamp_min(1e-12)
    return H / H.sum(-1, keepdim=True).clamp_min(1e-12)

Wds = sinkhorn(torch.randn(1, 1, 4, 4))
print("after Sinkhorn row/col err:", doubly_stochastic_error(Wds))

## 4. Why this stabilizes deep nets

Doubly stochastic matrices are **closed under multiplication**, so the composite product over many layers stays bounded. `scripts/plot_mhc_gain.py` reproduces this (see `assets/mhc_gain.png`).

In [ ]:
torch.manual_seed(0)
n = 4
P_hc, P_mhc = torch.eye(n), torch.eye(n)
gain_hc, gain_mhc = [], []
for _ in range(64):
    M = torch.randn(n, n)
    P_hc = M @ P_hc
    P_mhc = sinkhorn(M) @ P_mhc
    gain_hc.append(P_hc.abs().sum(-1).max().item())
    gain_mhc.append(P_mhc.abs().sum(-1).max().item())
print(f"composite gain @ depth 64:  HC={gain_hc[-1]:.2e}   mHC={gain_mhc[-1]:.3f}")

## Next steps
- `python train.py --config config/train_shakespeare_char.py` — 17-minute real training
- `python sample.py --ckpt runs/mhc/ckpt_final.pt --chat` — talk to it
- `tests/smoke.py` — the full verification suite